# AlphaBench EDA — return distributions, stationarity, correlation structure

**Educational research artifact — not investment advice.**

This notebook covers the Week 2 deliverable from [`docs/PROPOSAL.md`](../docs/PROPOSAL.md)
(§ weekly plan): return distributions and fat tails, volatility clustering, ADF/KPSS on
**prices vs returns**, correlation structure, and a regime timeline — followed by a written
note on what each implies for modelling.

Two conventions, both deliberate:

- **It imports from `src/`, never the reverse.** Per `docs/TECH_STACK_AND_STRUCTURE.md`,
  anything that matters lives in the tested package; this notebook only calls it.
- **Every figure is also written to `reports/figures/`.** The repo's pre-commit hook strips
  notebook outputs (keeps diffs reviewable and the repo small), so the committed `.ipynb`
  shows code without charts. Saving the PNGs means the visual results survive in git and can
  be viewed without running anything.

It runs entirely from committed data (`data/processed/features.parquet`) — no network, and
nothing here touches the sealed holdout or changes any reported number.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from alphabench.models.arima import acf_pacf, adf_test, kpss_test

FIG = Path("../reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

feats = pd.read_parquet("../data/processed/features.parquet")
feats["date"] = pd.to_datetime(feats["date"])
print(
    f"{len(feats):,} rows | {feats['symbol'].nunique()} symbols | "
    f"{feats['date'].min().date()} -> {feats['date'].max().date()}"
)

## 1. Return distributions and fat tails

Daily log returns, pooled across the universe, against a normal fitted to the same mean and
standard deviation. The question is not "are they roughly bell-shaped" (they are) but how
badly the tails misbehave — that is what decides whether Gaussian-flavoured assumptions
anywhere downstream are safe.

One caveat surfaces immediately, and is reported rather than smoothed over: the pooled
kurtosis is dominated by a *single* observation. Both numbers are shown below.

In [ ]:
rets = (
    feats.sort_values(["symbol", "date"])
    .groupby("symbol")["close"]
    .transform(lambda s: np.log(s).diff())
)
rets = rets.dropna()

mu, sigma = rets.mean(), rets.std()

# data/validate.py warns on single-day moves >50% as likely unadjusted corporate actions.
# Report the pooled statistics with and without them, because one row moves kurtosis by
# an order of magnitude and quoting only the headline figure would be misleading.
suspect = rets.abs() > 0.50
clean = rets[~suspect]

print(f"n={len(rets):,}  mean={mu:.6f}  sd={sigma:.4f}")
print(
    f"  all data          : skew={stats.skew(rets):>7.3f}   "
    f"excess kurtosis={stats.kurtosis(rets):>7.2f}"
)
print(
    f"  excl. {suspect.sum()} move(s) >50%: skew={stats.skew(clean):>7.3f}   "
    f"excess kurtosis={stats.kurtosis(clean):>7.2f}   <- the representative figure"
)
print(f"  (normal would be 0 for both; Jarque-Bera p={stats.jarque_bera(clean).pvalue:.2e})")

# How many big moves actually happen vs how many a normal would predict?
tail_rows = []
for k in (3, 4, 5, 6):
    actual = int((clean.abs() > k * sigma).sum())
    expected = float(2 * stats.norm.sf(k) * len(clean))
    tail_rows.append(
        {
            "threshold": f"|r| > {k}s",
            "observed": actual,
            "normal_expects": round(expected, 1),
            "ratio": round(actual / expected, 1) if expected > 0 else np.nan,
        }
    )
pd.DataFrame(tail_rows)

### The outlier, named

Worth looking at directly rather than leaving as "an outlier". The largest moves in the
panel:

In [ ]:
largest = feats.assign(ret=rets).loc[rets.abs().nlargest(6).index]
print(largest[["date", "symbol", "close", "ret"]].to_string(index=False))
print(
    "\nThe single >50% move is a ~-76% one-day log return in NESTLEIND.NS in Jan 2010 —"
    "\ndecisively an unadjusted corporate action rather than a real price move. It sits in"
    "\n2010, seven years before the first validation fold (2017), so it affects the training"
    "\nwindow only and never a scored prediction. It is left in place rather than hand-patched:"
    "\nsilently editing provider data is a worse habit than carrying one flagged row, and"
    "\nvalidate_panel already surfaces it on every ingest."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

x = np.linspace(rets.quantile(0.0005), rets.quantile(0.9995), 500)
axes[0].hist(rets, bins=200, density=True, alpha=0.65, label="observed")
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), "r--", lw=1.5, label="normal fit")
axes[0].set_yscale("log")
axes[0].set_xlabel("daily log return")
axes[0].set_ylabel("density (log scale)")
axes[0].set_title("Return distribution vs normal")
axes[0].legend()

stats.probplot(rets.sample(20000, random_state=42), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot vs normal (20k sample)")
axes[1].get_lines()[0].set_markersize(2)

fig.tight_layout()
fig.savefig(FIG / "eda_return_distribution.png", dpi=150)
plt.show()

## 2. Volatility clustering

The ACF of returns against the ACF of *absolute* returns. If returns are close to a
martingale difference sequence but volatility is persistent, the first should die
essentially immediately and the second should decay slowly.

In [ ]:
sym = sorted(feats["symbol"].unique())[0]
one = feats[feats["symbol"] == sym].sort_values("date")
r_one = np.log(one["close"]).diff().dropna()

ap_ret = acf_pacf(r_one, nlags=40)
ap_abs = acf_pacf(r_one.abs(), nlags=40)
ci = 1.96 / np.sqrt(len(r_one))  # approximate white-noise band

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, ap, title in (
    (axes[0], ap_ret, f"ACF of returns — {sym}"),
    (axes[1], ap_abs, f"ACF of |returns| — {sym}"),
):
    lags = range(1, len(ap["acf"]))
    ax.bar(lags, ap["acf"][1:], width=0.7)
    ax.axhline(ci, color="r", ls="--", lw=1)
    ax.axhline(-ci, color="r", ls="--", lw=1)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("lag (days)")
axes[0].set_ylabel("autocorrelation")

fig.tight_layout()
fig.savefig(FIG / "eda_volatility_clustering.png", dpi=150)
plt.show()

## 3. Stationarity — prices vs returns

The specific claim to test (PROPOSAL Week 2): **prices are non-stationary, returns are
stationary.** ADF and KPSS have opposite nulls, so the strongest evidence is when they
agree: ADF rejecting a unit root *and* KPSS failing to reject stationarity.

| | ADF null | KPSS null |
|---|---|---|
| H0 | series has a unit root (non-stationary) | series is stationary |
| "stationary" verdict | p < 0.05 (reject) | p > 0.05 (fail to reject) |

In [ ]:
rows = []
for s in sorted(feats["symbol"].unique()):
    g = feats[feats["symbol"] == s].sort_values("date")
    price = g["close"].dropna()
    ret = np.log(g["close"]).diff().dropna()
    if len(ret) < 250:
        continue
    rows.append(
        {
            "symbol": s,
            "price_adf_stationary": adf_test(price)["stationary_at_5pct"],
            "price_kpss_stationary": kpss_test(price)["stationary_at_5pct"],
            "ret_adf_stationary": adf_test(ret)["stationary_at_5pct"],
            "ret_kpss_stationary": kpss_test(ret)["stationary_at_5pct"],
        }
    )

stat = pd.DataFrame(rows)
n = len(stat)
print(f"Across {n} symbols:\n")
print(f"  PRICES   ADF says stationary : {stat['price_adf_stationary'].sum():>2}/{n}")
print(f"           KPSS says stationary: {stat['price_kpss_stationary'].sum():>2}/{n}")
print(f"  RETURNS  ADF says stationary : {stat['ret_adf_stationary'].sum():>2}/{n}")
print(f"           KPSS says stationary: {stat['ret_kpss_stationary'].sum():>2}/{n}")
print("\nExpected: prices non-stationary on both, returns stationary on both.")
stat.head(10)

## 4. Correlation structure

Two things worth knowing before modelling: how correlated the engineered features are with
each other (multicollinearity, and whether the ~50-feature set is really ~50 independent
things), and how correlated any of them are with the *forward* return — the latter being the
leakage tripwire the test suite enforces at 0.10.

In [ ]:
meta = {"date", "symbol", "open", "high", "low", "close", "volume"}
fcols = [c for c in feats.columns if c not in meta]

corr = feats[fcols].corr()
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"shrink": 0.6},
    ax=ax,
)
ax.tick_params(labelsize=6)
ax.set_title("Feature correlation matrix")
fig.tight_layout()
fig.savefig(FIG / "eda_feature_correlation.png", dpi=150)
plt.show()

# Most redundant pairs (upper triangle only, so each pair appears once)
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
top_pairs = (
    corr.abs()
    .where(mask)
    .rename_axis(index="feature_a")
    .melt(ignore_index=False, var_name="feature_b", value_name="abs_corr")
    .reset_index()
    .dropna()
    .nlargest(8, "abs_corr")
    .round(3)
)
print("Most collinear feature pairs:")
print(top_pairs.to_string(index=False))

In [ ]:
targets = pd.read_parquet("../data/processed/targets.parquet")
merged = feats.merge(targets, on=["date", "symbol"]).dropna(subset=["fwd_ret_1d"])
fwd_corr = merged[fcols].corrwith(merged["fwd_ret_1d"]).abs().sort_values(ascending=False)

print("Strongest |correlation| with the NEXT day's return:")
print(fwd_corr.head(10).round(4))
print(f"\nMax = {fwd_corr.max():.4f}. The leakage test fails the build above 0.10.")
print("Values this small are the honest signal level, not a bug.")

## 5. Regime timeline

Rolling 21-day annualised volatility, averaged across the universe. Useful context for the
per-year backtest breakdown in the technical report: it shows which years were calm and which
were not, so a year of poor performance can be checked against whether it was simply a
violent year.

In [ ]:
vol = (
    feats.sort_values(["symbol", "date"])
    .assign(r=lambda d: d.groupby("symbol")["close"].transform(lambda s: np.log(s).diff()))
    .assign(
        v=lambda d: d.groupby("symbol")["r"].transform(lambda s: s.rolling(21).std() * np.sqrt(252))
    )
    .groupby("date")["v"]
    .mean()
    .dropna()
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol.index, vol.to_numpy(), lw=0.9)
ax.axhline(vol.median(), color="r", ls="--", lw=1, label=f"median {vol.median():.1%}")
ax.set_ylabel("annualised vol (21d rolling)")
ax.set_title("Universe-average volatility regime")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "eda_volatility_regime.png", dpi=150)
plt.show()

print(vol.groupby(vol.index.year).mean().round(3).to_string())

## What this implies for modelling

The written note the proposal asks for. Five findings, and what each one forced:

**1. Returns are fat-tailed, not normal.** Excess kurtosis is ~9 once the single flagged
corporate-action row is set aside (~313 with it — which is itself the lesson that one bad
row can dominate a pooled moment), Jarque-Bera rejects normality outright, and moves beyond
4σ occur many times more often than a Gaussian allows. So no part of the pipeline leans on a
normality assumption for inference. The Sharpe confidence intervals come from a *block
bootstrap* rather than a closed-form standard error, and the deflated Sharpe carries
explicit skew/kurtosis terms.

**2. Prices are non-stationary; returns are stationary.** ADF and KPSS agree on both counts.
This is why every model in this project is fitted to returns, never to price levels — and it
is the direct reason ARIMA (B2) is specified on log returns with `d=0`. It is also why
"predict tomorrow's price" is the wrong problem: a model fitted to a random walk in levels
scores beautifully on RMSE by predicting "roughly today's price" and has learned nothing.

**3. Volatility clusters even though returns barely autocorrelate.** The ACF of returns dies
essentially at lag 1, while the ACF of |returns| decays slowly. Two consequences: the
*direction* of returns has little linear structure to exploit (consistent with the
noise-level AUCs this project reports), and the *magnitude* is predictable enough to be worth
using as a feature — hence the volatility block in the feature set, and the
volatility-scaled deadband on the labels, which stops the target from being dominated by
whatever the current vol regime happens to be.

**4. The feature set is far less than ~50 independent things.** Whole blocks of the
correlation matrix are near-collinear by construction (overlapping momentum lookbacks,
several volatility estimators of the same underlying quantity). This is why the primary model
is a heavily-regularised, shallow, small-leaf GBM: with this much redundancy and this little
signal, an unconstrained learner memorises noise. It is also why feature *importance* is read
through SHAP with a fold-stability check rather than trusted from a single fit.

**5. No feature correlates meaningfully with the forward return.** The strongest is a few
hundredths. That is the honest signal level for this problem, and it sets expectations
correctly: the ceiling here is a small edge, not a large one. The leakage test enforces this
from the other side — any feature exceeding 0.10 fails the build, because at this signal
level a "strong" predictor is far more likely to be a bug than a discovery.

Together these are the empirical case for the project's design: predict *direction* on a
volatility-scaled deadband, validate walk-forward with purging and embargo, report
distribution-free confidence intervals, and treat any large number as a bug until proven
otherwise.